In [1]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer numpy

  Using cached qiskit-2.3.0-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (12 kB)
  Using cached qiskit_ibm_runtime-0.45.1-py3-none-any.whl.metadata (21 kB)
  Using cached qiskit_aer-0.17.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.3 kB)
  Using cached rustworkx-0.17.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached requests_ntlm-1.3.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached ibm_platform_services-0.74.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached ibm_cloud_sdk_core-3.24.4-py3-none-any.whl.metadata (8.7 kB)
  Using cached pyspnego-0.12.1-py3-none-any.whl.metadata (4.1 kB)
Using cached qiskit-2.3.0-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.9 MB)
Using cached qiskit_ibm_runtime-0.45.1-py3-none-any.whl (1.5 MB)
Using cached qiskit_aer-0.17.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

import numpy as np

In [3]:
def match_operator(n: int, char_size: int):
    '''
    page 12 of the paper
    '''
    qx = QuantumRegister(n*char_size, 'x')
    qy = QuantumRegister(n*char_size, 'y')
    qout = QuantumRegister(n, 'o')
    qc = QuantumCircuit(qx, qy, qout)

    for i in range(n*char_size):
        qc.cx(qx[i], qy[i])
        qc.x(qy[i])

    for i in range(0, n):
        ctrls = list(range(i*char_size, i*char_size + char_size))
        qc.mcx([*qy[ctrls]], qout[i])
    return qc

qubits = 4
char_size = 3
qr = []

match_operator(qubits, char_size).draw()

»
 x_0: ──■───────────────────────────────────────────────────────────────────»
        │                                                                   »
 x_1: ──┼────■──────────────────────────────────────────────────────────────»
        │    │                                                              »
 x_2: ──┼────┼────■─────────────────────────────────────────────────────────»
        │    │    │                                                         »
 x_3: ──┼────┼────┼────■────────────────────────────────────────────────────»
        │    │    │    │                                                    »
 x_4: ──┼────┼────┼────┼────■───────────────────────────────────────────────»
        │    │    │    │    │                                               »
 x_5: ──┼────┼────┼────┼────┼────■──────────────────────────────────────────»
        │    │    │    │    │    │                                          »
 x_6: ──┼────┼────┼────┼────┼────┼────■─────────────────────────────────────»
        │    │    │    │    │    │    │                                     »
 x_7: ──┼────┼────┼────┼────┼────┼────┼────■────────────────────────────────»
        │    │    │    │    │    │    │    │                                »
 x_8: ──┼────┼────┼────┼────┼────┼────┼────┼────■───────────────────────────»
        │    │    │    │    │    │    │    │    │                           »
 x_9: ──┼────┼────┼────┼────┼────┼────┼────┼────┼────■──────────────────────»
        │    │    │    │    │    │    │    │    │    │                      »
x_10: ──┼────┼────┼────┼────┼────┼────┼────┼────┼────┼────■─────────────────»
        │    │    │    │    │    │    │    │    │    │    │                 »
x_11: ──┼────┼────┼────┼────┼────┼────┼────┼────┼────┼────┼────■────────────»
      ┌─┴─┐  │    │    │    │    │    │    │    │    │    │    │  ┌───┐     »
 y_0: ┤ X ├──┼────┼────┼────┼────┼────┼────┼────┼────┼────┼────┼──┤ X ├──■──»
      └───┘┌─┴─┐  │    │    │    │    │    │    │    │    │    │  ├───┤  │  »
 y_1: ─────┤ X ├──┼────┼────┼────┼────┼────┼────┼────┼────┼────┼──┤ X ├──■──»
           └───┘┌─┴─┐  │    │    │    │    │    │    │    │    │  ├───┤  │  »
 y_2: ──────────┤ X ├──┼────┼────┼────┼────┼────┼────┼────┼────┼──┤ X ├──■──»
                └───┘┌─┴─┐  │    │    │    │    │    │    │    │  ├───┤  │  »
 y_3: ───────────────┤ X ├──┼────┼────┼────┼────┼────┼────┼────┼──┤ X ├──┼──»
                     └───┘┌─┴─┐  │    │    │    │    │    │    │  ├───┤  │  »
 y_4: ────────────────────┤ X ├──┼────┼────┼────┼────┼────┼────┼──┤ X ├──┼──»
                          └───┘┌─┴─┐  │    │    │    │    │    │  ├───┤  │  »
 y_5: ─────────────────────────┤ X ├──┼────┼────┼────┼────┼────┼──┤ X ├──┼──»
                               └───┘┌─┴─┐  │    │    │    │    │  ├───┤  │  »
 y_6: ──────────────────────────────┤ X ├──┼────┼────┼────┼────┼──┤ X ├──┼──»
                                    └───┘┌─┴─┐  │    │    │    │  ├───┤  │  »
 y_7: ───────────────────────────────────┤ X ├──┼────┼────┼────┼──┤ X ├──┼──»
                                         └───┘┌─┴─┐  │    │    │  ├───┤  │  »
 y_8: ────────────────────────────────────────┤ X ├──┼────┼────┼──┤ X ├──┼──»
                                              └───┘┌─┴─┐  │    │  ├───┤  │  »
 y_9: ─────────────────────────────────────────────┤ X ├──┼────┼──┤ X ├──┼──»
                                                   └───┘┌─┴─┐  │  ├───┤  │  »
y_10: ──────────────────────────────────────────────────┤ X ├──┼──┤ X ├──┼──»
                                                        └───┘┌─┴─┐├───┤  │  »
y_11: ───────────────────────────────────────────────────────┤ X ├┤ X ├──┼──»
                                                             └───┘└───┘┌─┴─┐»
 o_0: ─────────────────────────────────────────────────────────────────┤ X ├»
                                                                       └───┘»
 o_1: ──────────────────────────────────────────────────────────────────────»
                    

In [4]:
def extension_operator(n: int, i: int):
    '''
    page 12 of the paper
    '''
    shift = 2 ** (i - 1)

    qin = QuantumRegister(n, 'in')
    qout = QuantumRegister(n, 'out')
    qc = QuantumCircuit(qin, qout)

    for j in range(n - shift):
        qc.ccx(qin[j], qin[j + shift], qout[j])

    return qc

qubits = 8
i = 2
qc = extension_operator(qubits,i)
qc.draw()

in_0: ──■───────────────────────────
         │                           
 in_1: ──┼────■──────────────────────
         │    │                      
 in_2: ──■────┼────■─────────────────
         │    │    │                 
 in_3: ──┼────■────┼────■────────────
         │    │    │    │            
 in_4: ──┼────┼────■────┼────■───────
         │    │    │    │    │       
 in_5: ──┼────┼────┼────■────┼────■──
         │    │    │    │    │    │  
 in_6: ──┼────┼────┼────┼────■────┼──
         │    │    │    │    │    │  
 in_7: ──┼────┼────┼────┼────┼────■──
       ┌─┴─┐  │    │    │    │    │  
out_0: ┤ X ├──┼────┼────┼────┼────┼──
       └───┘┌─┴─┐  │    │    │    │  
out_1: ─────┤ X ├──┼────┼────┼────┼──
            └───┘┌─┴─┐  │    │    │  
out_2: ──────────┤ X ├──┼────┼────┼──
                 └───┘┌─┴─┐  │    │  
out_3: ───────────────┤ X ├──┼────┼──
                      └───┘┌─┴─┐  │  
out_4: ────────────────────┤ X ├──┼──
                           └───┘┌─┴─┐
out_5: ─────────────────────────┤ X ├
                                └───┘
out_6: ──────────────────────────────
                                     
out_7: ──────────────────────────────

### **Operatori di rotazione**

Fondamentali nel contesto degli algoritmi di ricerca su stringhe basati su Grover.

In [5]:
def quantum_rotation(n: int, s: int):
    qr = QuantumRegister(n, 'q')
    qc = QuantumCircuit(qr)

    for i in range(1, int(np.ceil(n/2))):
        qc.swap(qr[i], qr[n-i])

    for j in range(1, int(np.ceil(n/2))):
        qc.swap(qr[int(np.ceil(s/2))-j], qr[int(np.floor(s/2))+j])

    return qc.to_gate(label='R_'+str(s)+' ')

qubits = 8
i = 2
qr = []
qr.append(QuantumRegister(qubits, 'q'))

qc = QuantumCircuit(*qr)
qc = qc.compose(quantum_rotation(qubits,i)).decompose()
qc.draw()

q_0: ──────────X────
               │    
q_1: ───────X──┼────
            │  │    
q_2: ────X──┼──X────
         │  │       
q_3: ─X──┼──┼─────X─
      │  │  │     │ 
q_4: ─┼──┼──┼──X──┼─
      │  │  │  │  │ 
q_5: ─X──┼──┼──┼──┼─
         │  │  │  │ 
q_6: ────X──┼──X──┼─
            │     │ 
q_7: ───────X─────X─

In [6]:
def quantum_parametric_rotation(n: int):

    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, 'j')
    qr = QuantumRegister(n, 'q')

    qc = QuantumCircuit(jr, qr)
    for i in range(l):

        qc = qc.compose(quantum_rotation(n, 2**i).control(1), [jr[i],*qr])
    return qc

qubits = 16

qc = quantum_parametric_rotation(4)
qc.draw()


j_0: ────■─────────────
         │             
j_1: ────┼────────■────
     ┌───┴───┐┌───┴───┐
q_0: ┤0      ├┤0      ├
     │       ││       │
q_1: ┤1      ├┤1      ├
     │  R_1  ││  R_2  │
q_2: ┤2      ├┤2      ├
     │       ││       │
q_3: ┤3      ├┤3      ├
     └───────┘└───────┘

In [7]:
def quantum_bounded_rotation(n: int, char_size: int):
    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, 'j')
    qr = QuantumRegister(n * char_size, 'q')

    qc = QuantumCircuit(jr, qr)

    for i in range(l):
        controlled_rot = quantum_rotation(n, 2**i).control(1)

        for bit in range(char_size):
            lines = [qr[k * char_size + bit] for k in range(n)]
            qc = qc.compose(controlled_rot, [jr[i], *lines])

    return qc

qc = quantum_bounded_rotation(8, 2)
qc.draw()


j_0: ────■────────■────────────────────────────────────────
          │        │                                        
 j_1: ────┼────────┼────────■────────■──────────────────────
          │        │        │        │                      
 j_2: ────┼────────┼────────┼────────┼────────■────────■────
      ┌───┴───┐    │    ┌───┴───┐    │    ┌───┴───┐    │    
 q_0: ┤0      ├────┼────┤0      ├────┼────┤0      ├────┼────
      │       │┌───┴───┐│       │┌───┴───┐│       │┌───┴───┐
 q_1: ┤       ├┤0      ├┤       ├┤0      ├┤       ├┤0      ├
      │       ││       ││       ││       ││       ││       │
 q_2: ┤1      ├┤       ├┤1      ├┤       ├┤1      ├┤       ├
      │       ││       ││       ││       ││       ││       │
 q_3: ┤       ├┤1      ├┤       ├┤1      ├┤       ├┤1      ├
      │       ││       ││       ││       ││       ││       │
 q_4: ┤2      ├┤       ├┤2      ├┤       ├┤2      ├┤       ├
      │       ││       ││       ││       ││       ││       │
 q_5: ┤       ├┤2      ├┤       ├┤2      ├┤       ├┤2      ├
      │       ││       ││       ││       ││       ││       │
 q_6: ┤3      ├┤       ├┤3      ├┤       ├┤3      ├┤       ├
      │       ││       ││       ││       ││       ││       │
 q_7: ┤  R_1  ├┤3      ├┤  R_2  ├┤3      ├┤  R_4  ├┤3      ├
      │       ││       ││       ││       ││       ││       │
 q_8: ┤4      ├┤  R_1  ├┤4      ├┤  R_2  ├┤4      ├┤  R_4  ├
      │       ││       ││       ││       ││       ││       │
 q_9: ┤       ├┤4      ├┤       ├┤4      ├┤       ├┤4      ├
      │       ││       ││       ││       ││       ││       │
q_10: ┤5      ├┤       ├┤5      ├┤       ├┤5      ├┤       ├
      │       ││       ││       ││       ││       ││       │
q_11: ┤       ├┤5      ├┤       ├┤5      ├┤       ├┤5      ├
      │       ││       ││       ││       ││       ││       │
q_12: ┤6      ├┤       ├┤6      ├┤       ├┤6      ├┤       ├
      │       ││       ││       ││       ││       ││       │
q_13: ┤       ├┤6      ├┤       ├┤6      ├┤       ├┤6      ├
      │       ││       ││       ││       ││       ││       │
q_14: ┤7      ├┤       ├┤7      ├┤       ├┤7      ├┤       ├
      └───────┘│       │└───────┘│       │└───────┘│       │
q_15: ─────────┤7      ├─────────┤7      ├─────────┤7      ├
               └───────┘         └───────┘         └───────┘

### **Operatore di "copia"**

Con "copia", ci riferiamo alla copia di stati base $|0\rangle$ e $|1\rangle$, nel totale rispetto (a malincuore?) del teorema di non-clonazione. Lo renderemo un operatore controllato tramite il metodo `.control(int)` offerto da Qiskit.

In [8]:
def copy_operator(n: int):

    qin = QuantumRegister(n, 'in')
    qout = QuantumRegister(n, 'out')
    qc = QuantumCircuit(qin, qout)

    for i in range(n):
        qc.cx(qin[i], qout[i])

    return qc.to_gate(label='COPY')

qubits = 16
qr = []
qr.append(QuantumRegister(qubits, 'q'))

qc = QuantumCircuit(*qr)
qc = qc.compose(copy_operator(qubits//2)).decompose()
qc.draw()

q_0: ──■─────────────────────────────────────
        │                                     
 q_1: ──┼────■────────────────────────────────
        │    │                                
 q_2: ──┼────┼────■───────────────────────────
        │    │    │                           
 q_3: ──┼────┼────┼────■──────────────────────
        │    │    │    │                      
 q_4: ──┼────┼────┼────┼────■─────────────────
        │    │    │    │    │                 
 q_5: ──┼────┼────┼────┼────┼────■────────────
        │    │    │    │    │    │            
 q_6: ──┼────┼────┼────┼────┼────┼────■───────
        │    │    │    │    │    │    │       
 q_7: ──┼────┼────┼────┼────┼────┼────┼────■──
      ┌─┴─┐  │    │    │    │    │    │    │  
 q_8: ┤ X ├──┼────┼────┼────┼────┼────┼────┼──
      └───┘┌─┴─┐  │    │    │    │    │    │  
 q_9: ─────┤ X ├──┼────┼────┼────┼────┼────┼──
           └───┘┌─┴─┐  │    │    │    │    │  
q_10: ──────────┤ X ├──┼────┼────┼────┼────┼──
                └───┘┌─┴─┐  │    │    │    │  
q_11: ───────────────┤ X ├──┼────┼────┼────┼──
                     └───┘┌─┴─┐  │    │    │  
q_12: ────────────────────┤ X ├──┼────┼────┼──
                          └───┘┌─┴─┐  │    │  
q_13: ─────────────────────────┤ X ├──┼────┼──
                               └───┘┌─┴─┐  │  
q_14: ──────────────────────────────┤ X ├──┼──
                                    └───┘┌─┴─┐
q_15: ───────────────────────────────────┤ X ├
                                         └───┘

### **Operatore OR logico**

Necessario per verificare se l'ultimo array $D[i]_j$ contiene almeno un elemento $=1$.

In [9]:
def or_operator(n: int):
    '''
    page 15 of the paper
    '''
    qx = QuantumRegister(n, 'x')
    qy = QuantumRegister(1, 'y')
    qc = QuantumCircuit(qx, qy)

    qc.x(qx)
    qc.mcx(qx, qy)
    qc.x(qx)
    qc.x(qy)

    return qc.to_gate(label='OR')

qubits = 5
qr = []
qr.append(QuantumRegister(qubits, 'q'))

qc = QuantumCircuit(qubits)
qc = qc.compose(or_operator(qubits-1)).decompose()
qc.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     └───┘┌─┴─┐├───┤
q_4: ─────┤ X ├┤ X ├
          └───┘└───┘

### **Operatore AND logico bit-a-bit controllato**

In [10]:
def contr_bitwise_and_operator(n: int):
    '''
    page 13 of the paper
    '''
    qc_ctrl = QuantumRegister(1, 'c')
    qL = QuantumRegister(n, 'L')
    qin = QuantumRegister(n+1, 'qin')
    qout  = QuantumRegister(n+1, 'qout')
    qc = QuantumCircuit(qc_ctrl, qL, qin, qout)

    for k in range(n):
        qc.mcx([qc_ctrl[0], qL[k], qin[k]], qout[k])

    return qc.to_gate(label="C-AND")

qubits = 5
qr = []
qr.append(QuantumRegister(qubits, 'q'))

qc = QuantumCircuit(15)
qc = qc.compose(contr_bitwise_and_operator(4)).decompose()
qc.draw()

q_0: ──■────■────■────■──
        │    │    │    │  
 q_1: ──■────┼────┼────┼──
        │    │    │    │  
 q_2: ──┼────■────┼────┼──
        │    │    │    │  
 q_3: ──┼────┼────■────┼──
        │    │    │    │  
 q_4: ──┼────┼────┼────■──
        │    │    │    │  
 q_5: ──■────┼────┼────┼──
        │    │    │    │  
 q_6: ──┼────■────┼────┼──
        │    │    │    │  
 q_7: ──┼────┼────■────┼──
        │    │    │    │  
 q_8: ──┼────┼────┼────■──
        │    │    │    │  
 q_9: ──┼────┼────┼────┼──
      ┌─┴─┐  │    │    │  
q_10: ┤ X ├──┼────┼────┼──
      └───┘┌─┴─┐  │    │  
q_11: ─────┤ X ├──┼────┼──
           └───┘┌─┴─┐  │  
q_12: ──────────┤ X ├──┼──
                └───┘┌─┴─┐
q_13: ───────────────┤ X ├
                     └───┘
q_14: ────────────────────

### **Operatore FSM completo**

In [11]:
def FSM(n, char_size):
    '''
    builds the Fixed Substring Matching operator for inputs of size `n`
    '''

    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n*char_size, 'x')
    qy = QuantumRegister(n*char_size, 'y')
    qd = QuantumRegister(d_len, 'd')
    qD = QuantumRegister(d_len+1, 'D')
    qL = []

    for i in range(L_num): # padding
        qL.append(QuantumRegister(n, 'L['+str(i)+']'))

    qD = []
    for i in range(d_len+1):
        qD.append(QuantumRegister(n+1, 'D['+str(i-1)+']'))

    qr = QuantumRegister(1, 'r')
    qc = QuantumCircuit(qx,qy,qd,*qL,*qD,qr)


    qc = qc.compose(match_operator(n, char_size).to_gate(label='MATCH'), [*qx, *qy, *qL[0]])

    for i in range(L_num-1):
        qc = qc.compose(extension_operator(n, i+1).to_gate(label=f'EXT_{i}'), [*qL[i][:n], *qL[i+1]])

    for i in range(d_len):

        qc = qc.compose(contr_bitwise_and_operator(n), [qd[i], *qL[i], *qD[i], *qD[i+1]])

        controlled_rot = quantum_rotation(n+1, 2**i).control(1)
        qc = qc.compose(controlled_rot, [qd[i], *qD[i+1]])
        qc.x(qd[i])
        controlled_copy = copy_operator(n+1).control(1)
        qc = qc.compose(controlled_copy, [qd[i],*qD[i], *qD[i+1]])
        qc.x(qd[i])

    qc = qc.compose(or_operator(n+1), [*qD[d_len], qr])

    return qc

qc = FSM(8, 2)
print(qc.num_qubits)
qc.draw()


96


┌─────────┐                                                         »
    x_0: ┤0        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_1: ┤1        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_2: ┤2        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_3: ┤3        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_4: ┤4        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_5: ┤5        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_6: ┤6        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_7: ┤7        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_8: ┤8        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    x_9: ┤9        ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_10: ┤10       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_11: ┤11       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_12: ┤12       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_13: ┤13       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_14: ┤14       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
   x_15: ┤15       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_0: ┤16       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_1: ┤17       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_2: ┤18       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_3: ┤19       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_4: ┤20       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_5: ┤21 MATCH ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_6: ┤22       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_7: ┤23       ├─────────────────────────────────────────────────────────»
         │         │                                                         »
    y_8: ┤24       ├─────────────────────────────────────────────────────────»
         │         │                                       

Da questo operatore, andiamo a creare i precedentemente citati $\text{FPM}$ e $\text{SFC}$.
Il primo nasce ponendo $D[-1]_0 = 1$ (che nel paper è $D^{-1}_0$). Questo, impone che la sottostringa comune di dimensione $d$ sia il prefisso.
Il secondo, invece, generalizza l'inizio della sottostringa comune a qualsiasi indice $j$.


In [12]:
def SFC(n, char_size):
    '''
    builds the Shared Fix Substring Check operator for inputs of size `n`
    '''
    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n*char_size, 'x')
    qy = QuantumRegister(n*char_size, 'y')
    qd = QuantumRegister(d_len, 'd')
    qD = QuantumRegister(d_len+1, 'D')
    qL = []

    for i in range(L_num): # padding
        qL.append(QuantumRegister(n, 'L['+str(i)+']'))

    qD = []
    for i in range(d_len+1):
        qD.append(QuantumRegister(n+1, 'D['+str(i-1)+']'))

    qr = QuantumRegister(1, 'r')
    qc = QuantumCircuit(qx,qy,qd,*qL,*qD,qr)

    qc.x(qD[0])

    qc = qc.compose(FSM(n, char_size))
    return qc

qc = SFC(4, 1)
qc.draw()

┌─────────┐                                                       »
    x_0: ┤0        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_1: ┤1        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_2: ┤2        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_3: ┤3        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_0: ┤4        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_1: ┤5        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_2: ┤6        ├───────────────────────────────────────────────────────»
         │   MATCH │                                                       »
    y_3: ┤7        ├───────────────────────────────────────────────────────»
         │         │          ┌─────────┐         ┌───┐            ┌───┐   »
    d_0: ┤         ├──────────┤0        ├────■────┤ X ├────■───────┤ X ├───»
         │         │          │         │    │    └───┘    │    ┌──┴───┴──┐»
    d_1: ┤         ├──────────┤         ├────┼─────────────┼────┤0        ├»
         │         │┌────────┐│         │    │             │    │         │»
 L[0]_0: ┤8        ├┤0       ├┤1        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_1: ┤9        ├┤1       ├┤2        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_2: ┤10       ├┤2       ├┤3        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_3: ┤11       ├┤3       ├┤4        ├────┼─────────────┼────┤         ├»
         └─────────┘│  EXT_0 ││         │    │             │    │         │»
 L[1]_0: ───────────┤4       ├┤         ├────┼─────────────┼────┤1        ├»
                    │        ││         │    │             │    │         │»
 L[1]_1: ───────────┤5       ├┤         ├────┼─────────────┼────┤2        ├»
                    │        ││         │    │             │    │         │»
 L[1]_2: ───────────┤6       ├┤         ├────┼─────────────┼────┤3        ├»
                    │        ││         │    │             │    │         │»
 L[1]_3: ───────────┤7       ├┤         ├────┼─────────────┼────┤4        ├»
            ┌───┐   └────────┘│   C-AND │    │         ┌───┴───┐│         │»
D[-1]_0: ───┤ X ├─────────────┤5        ├────┼─────────┤0      ├┤         ├»
            ├───┤             │         │    │         │       ││         │»
D[-1]_1: ───┤ X ├─────────────┤6        ├────┼─────────┤1      ├┤         ├»
            ├───┤             │         │    │         │       ││         │»
D[-1]_2: ───┤ X ├─────────────┤7        ├────┼─────────┤2      ├┤         ├»
            ├───┤             │         │    │         │       ││   C-AND │»
D[-1]_3: ───┤ X ├─────────────┤8        ├────┼─────────┤3      ├┤         ├»
            ├───┤             │         │    │         │       ││         │»
D[-1]_4: ───┤ X ├─────────────┤9        ├────┼─────────┤4      ├┤         ├»
            └───┘             │         │┌───┴───┐     │  COPY ││         │»
 D[0]_0: ─────────────────────┤10       ├┤0      ├─────┤5      ├┤5        ├»
                              │         ││       │     │       ││         │»
 D[0]_1: ─────────────────────┤11       ├┤1      ├─────┤6      ├┤6        ├»
                              │         ││       │     │       ││         │»
 D[0]_2: ─────────────────────┤12       ├┤2 R_1  ├─────┤7      ├┤7        ├»
     

In [13]:
def FPM(n, char_size):
    '''
    builds the Fixed Prefix Matching operator for inputs of size `n`
    '''
    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n*char_size, 'x')
    qy = QuantumRegister(n*char_size, 'y')
    qd = QuantumRegister(d_len, 'd')
    qD = QuantumRegister(d_len+1, 'D')
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, 'L['+str(i)+']'))

    qD = []
    for i in range(d_len+1):
        qD.append(QuantumRegister(n+1, 'D['+str(i-1)+']'))

    qr = QuantumRegister(1, 'r')
    qc = QuantumCircuit(qx,qy,qd,*qL,*qD,qr)

    # init
    qc.x(qD[0][0])

    qc = qc.compose(FSM(n, char_size))
    return qc

qc = FPM(4, 2)
qc.draw()

┌─────────┐                                                       »
    x_0: ┤0        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_1: ┤1        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_2: ┤2        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_3: ┤3        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_4: ┤4        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_5: ┤5        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_6: ┤6        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    x_7: ┤7        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_0: ┤8        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_1: ┤9        ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_2: ┤10       ├───────────────────────────────────────────────────────»
         │   MATCH │                                                       »
    y_3: ┤11       ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_4: ┤12       ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_5: ┤13       ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_6: ┤14       ├───────────────────────────────────────────────────────»
         │         │                                                       »
    y_7: ┤15       ├───────────────────────────────────────────────────────»
         │         │          ┌─────────┐         ┌───┐            ┌───┐   »
    d_0: ┤         ├──────────┤0        ├────■────┤ X ├────■───────┤ X ├───»
         │         │          │         │    │    └───┘    │    ┌──┴───┴──┐»
    d_1: ┤         ├──────────┤         ├────┼─────────────┼────┤0        ├»
         │         │┌────────┐│         │    │             │    │         │»
 L[0]_0: ┤16       ├┤0       ├┤1        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_1: ┤17       ├┤1       ├┤2        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_2: ┤18       ├┤2       ├┤3        ├────┼─────────────┼────┤         ├»
         │         ││        ││         │    │             │    │         │»
 L[0]_3: ┤19       ├┤3       ├┤4        ├────┼─────────────┼────┤         ├»
         └─────────┘│  EXT_0 ││         │    │             │    │         │»
 L[1]_0: ───────────┤4       ├┤         ├────┼─────────────┼────┤1        ├»
                    │        ││         │    │             │    │         │»
 L[1]_1: ───────────┤5       ├┤         ├────┼─────────────┼────┤2        ├»
                    │        ││         │    │             │    │         │»
 L[1]_2: ───────────┤6       ├┤         ├────┼─────────────┼────┤3        ├»
                    │        ││         │    │             │    │         │»
 L[1]_3: ───────────┤7       ├┤         ├────┼─────────────┼────┤4        ├»
     

## **Costruiamo il test quantistico**

In [14]:
def grover_operator(n: int):
    '''
    builds the diffusion operator from the grover algorithm
    '''
    qx = QuantumRegister(n, 'x')
    qc = QuantumCircuit(qx)

    qc.h(qx)
    qc.x(qx)

    if n == 1:
        qc.z(qx[0])
    else:
        qc.h(qx[n-1])
        qc.mcx(qx[list(range(n-1))], qx[n-1])
        qc.h(qx[n-1])
    qc.x(qx)
    qc.h(qx)

    return qc.to_gate(label='DIFFUSION')


In [15]:
def int_to_bin(num: int, code_len: int):
    """
    converts `num` in a binary string, with the specified `code_len`
    """
    num = bin(num)[2:]
    num = '0'*(code_len-len(num)) + num
    return num



In [16]:
def quantum_number_encode(num: int, code_len: int=0):
    """
    encodes the given `num` into a quantum circuit. if specified, `code_len` lets
    you use more qubits than needed for the number specified by `num`
    """
    if num == 0:
        return QuantumCircuit(code_len).to_gate(label=" 0 ")

    if code_len < int(np.ceil(np.log2(num))):
        code_len = len(bin(num)[2:])
    bin_num = int_to_bin(num, code_len)
    qc = QuantumCircuit(code_len)
    for c, i in enumerate(bin_num):
        if i == '1':
            qc.x(code_len-1-c)
    return qc.to_gate(label=" "+str(num)+" ")


In [17]:
def encode_boolean_string(x: str):
    n = len(x)
    qx = QuantumRegister(n, 'x')
    qc = QuantumCircuit(qx)

    for c, i in enumerate(x):
        if i == '1':
            qc.x(qx[c])

    return qc

Il test quantistico si compone quindi di tre fasi:
- $\text{SEARCH PHASE}$: «Esiste qualche posizione $i$ in cui c'è un match di lunghezza $d?$». In questo modo viene fissata la rotazione della stringa $X$ rispetto al match.
- $\text{VERIFICATION PHASE}$: «Ruotate in maniera opportuna, le stringhe matchano partendo dalla posizione 0?». In questo modo viene fissata la rotazione della stringa $Y$.
- $\text{FINAL CHECK}$: viene verificata la condizione tramite oracolo booleano. In questo modo, non è possibile avere falsi positivi. I falsi negativi vengono mitigati aumentando il numero di esecuzioni, in quanto basta anche solo un'esecuzione con output 1 per rendere il test valido.

In [28]:
def quantum_step_circuit(n: int, d: int, x: str, y: str, char_size: int):

    qi_amount = int(np.ceil(np.log2(n)))
    qi = QuantumRegister(qi_amount, 'i')
    qj = QuantumRegister(qi_amount, 'j')
    qx = QuantumRegister(n*char_size, 'x')
    qy = QuantumRegister(n*char_size, 'y')
    ancilla_amount = n * int(np.ceil(np.log2(n))) + (n + 1)*(int(np.ceil(np.log2(n + 1))))
    qa = QuantumRegister(ancilla_amount, 'a')
    qd = QuantumRegister(qi_amount, 'd')
    qr = QuantumRegister(1, 'r')
    qo = QuantumRegister(1, 'out')
    c = ClassicalRegister(1,'c')

    qc = QuantumCircuit(qi,qj,qx,qy,qa,qd,qr,qo, c)

    qc = qc.compose(quantum_number_encode(d,qi_amount), qd)
    qc = qc.compose(encode_boolean_string(x), qx)
    qc = qc.compose(encode_boolean_string(y), qy)
    qc.barrier(label='init')

    # search phase
    qc.h(qj)
    qc.x(qo)
    qc.h(qo)

    epochs = int(np.floor(np.pi / 4 * np.sqrt(n)))
    epochs = max(1, int(np.floor(np.pi / 4 * np.sqrt(n * char_size/d)))) # proporzionalità inversa rispetto alla lunghezza della stringa. DEVO testare una proporzionalità rispetto alla dimensione della codifica

    for _ in range(epochs):
        qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qj,*qx])
        qc = qc.compose(SFC(n, char_size).to_gate(label='SFC'), [*qx, *qy, *qd, *qa, qr])

        qc.cx(qr, qo)
        qc = qc.compose(SFC(n, char_size).inverse().to_gate(label='- SFC'), [*qx, *qy, *qd, *qa, qr])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qj,*qx])

        qc = qc.compose(grover_operator(qi_amount), qj)

    for j in qj:
        qc.measure(j, c)


    qc.barrier(label='SEARCH PHASE')

    qc.h(qi)

    # verification phase

    for _ in range(epochs):
        qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qj,*qx])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qi,*qx])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qi,*qy])
        qc = qc.compose(FPM(n, char_size).to_gate(label='FPM'), [*qx, *qy, *qd, *qa, qr])

        qc.cx(qr, qo)
        qc = qc.compose(FPM(n, char_size).inverse().to_gate(label='- FPM'), [*qx, *qy, *qd, *qa, qr])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qi,*qy])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qi,*qx])
        qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qj,*qx])

        qc = qc.compose(grover_operator(qi_amount), qi)

    for i in qi:
        qc.measure(i, c)
    qc.barrier(label='VERIFICATION PHASE')

    # final check
    qc.h(qo)
    qc.x(qo)

    qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qj,*qx])
    qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qi,*qx])
    qc = qc.compose(quantum_bounded_rotation(n, char_size).to_gate(label='ROT'), [*qi,*qy])
    qc = qc.compose(FPM(n, char_size).to_gate(label='FPM'), [*qx, *qy, *qd, *qa, qr])
    qc.cx(qr, qo)
    qc = qc.compose(FPM(n, char_size).inverse().to_gate(label='- FPM'), [*qx, *qy, *qd, *qa, qr])
    qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qi,*qy])
    qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qi,*qx])
    qc = qc.compose(quantum_bounded_rotation(n, char_size).inverse().to_gate(label='- ROT'), [*qj,*qx])

    qc.measure(qo, c)
    qc.barrier(label='FINAL CHECK')

    return qc

quantum_step_circuit(2, 1, '0010', '0000', 2).draw()

init                                                              »
  i: ─────────░────────────────────────────────────────────────────────────────»
              ░   ┌───┐┌──────┐                         ┌────────┐┌───────────┐»
  j: ─────────░───┤ H ├┤0     ├─────────────────────────┤0       ├┤ DIFFUSION ├»
              ░   └───┘│      │┌───────┐     ┌─────────┐│        │└───────────┘»
x_0: ─────────░────────┤1     ├┤0      ├─────┤0        ├┤1       ├─────────────»
              ░        │      ││       │     │         ││        │             »
x_1: ─────────░────────┤2 ROT ├┤1      ├─────┤1        ├┤2 - ROT ├─────────────»
      ┌───┐   ░        │      ││       │     │         ││        │             »
x_2: ─┤ X ├───░────────┤3     ├┤2      ├─────┤2        ├┤3       ├─────────────»
      └───┘   ░        │      ││       │     │         ││        │             »
x_3: ─────────░────────┤4     ├┤3      ├─────┤3        ├┤4       ├─────────────»
              ░        └──────┘│       │     │         │└────────┘             »
y_0: ─────────░────────────────┤4      ├─────┤4        ├───────────────────────»
              ░                │       │     │         │                       »
y_1: ─────────░────────────────┤5      ├─────┤5        ├───────────────────────»
              ░                │       │     │         │                       »
y_2: ─────────░────────────────┤6      ├─────┤6        ├───────────────────────»
              ░                │       │     │         │                       »
y_3: ─────────░────────────────┤7      ├─────┤7        ├───────────────────────»
              ░                │       │     │         │                       »
a_0: ─────────░────────────────┤9      ├─────┤9        ├───────────────────────»
              ░                │   SFC │     │   - SFC │                       »
a_1: ─────────░────────────────┤10     ├─────┤10       ├───────────────────────»
              ░                │       │     │         │                       »
a_2: ─────────░────────────────┤11     ├─────┤11       ├───────────────────────»
              ░                │       │     │         │                       »
a_3: ─────────░────────────────┤12     ├─────┤12       ├───────────────────────»
              ░                │       │     │         │                       »
a_4: ─────────░────────────────┤13     ├─────┤13       ├───────────────────────»
              ░                │       │     │         │                       »
a_5: ─────────░────────────────┤14     ├─────┤14       ├───────────────────────»
              ░                │       │     │         │                       »
a_6: ─────────░────────────────┤15     ├─────┤15       ├───────────────────────»
              ░                │       │     │         │                       »
a_7: ─────────░────────────────┤16     ├─────┤16       ├───────────────────────»
     ┌─────┐  ░                │       │     │         │                       »
  d: ┤  1  ├──░────────────────┤8      ├─────┤8        ├───────────────────────»
     └─────┘  ░                │       │     │         │                       »
  r: ─────────░────────────────┤17     ├──■──┤17       ├───────────────────────»
              ░   ┌───┐ ┌───┐  └───────┘┌─┴─┐└─────────┘                       »
out: ─────────░───┤ X ├─┤ H ├───────────┤ X ├──────────────────────────────────»
              ░   └───┘ └───┘           └───┘                                  »
c: 1/══════════════════════════════════════════════════════════════════════════»
                                                                               »
«         SEARCH PHASE  ┌───┐  ┌──────┐┌──────┐                         »
«  i: ─────────░────────┤ H ├──┤0     ├┤0     ├─────────────────────────»
«     ┌─┐      ░       ┌┴───┴─┐│      ││      │                         »
«  j: ┤M├──────░───────┤0     ├┤      ├┤      ├─────────────────────────»
«     └╥┘      ░       │      ││      ││      │┌───────┐     ┌─────────┐

In [19]:
def run(qc, shots=1024):
    """
    ritorna i risultati dell'esecuzione del `qc` `QuantumCircuit` nel simulatore ideale AerSimulator.
    `shots` permette di specificare il numero di esecuzioni da compiere.
    """
    sim_backend = AerSimulator(method='matrix_product_state')
    sim_backend.set_max_qubits(400)
    pm = generate_preset_pass_manager(backend=sim_backend, optimization_level=2)
    isa_qc = pm.run(qc)
    sampler = Sampler(mode=sim_backend)

    job = sampler.run([isa_qc], shots=shots)
    results = job.result()
    data = results[0].data
    counts = data.c.get_counts()

    return counts


In [29]:
accuracy = 25

def quantum_LCS_test(n: int, d: int, x: str, y: str, char_size: int):

    if d == 0:
        return True

    if d == n:
        return x == y

    qc = quantum_step_circuit(n, d, x, y, char_size)
    c = run(qc, accuracy)
    print(c)

    if '1' in list(c.keys()):
        return True
    else:
        return False


In [30]:
def quantum_LCS(x: str, y: str, n: int, char_size: int, verbose=False):
    l = 0
    r = n
    while l < r:
        d = (l + r + 1) // 2
        if verbose:
            print(f"testing for common substring of length {d}:", end=" ")

        if quantum_LCS_test(n, d, x, y, char_size):
            l = d
            if verbose: print("found!")
        else:
            r = d - 1
            if verbose: print("not found!")
    return l


In [22]:
X = '11111111'
Y = '11111111'

print(quantum_LCS(X, Y, 4, 2, True))

testing for common substring of length 2: found!
testing for common substring of length 3: found!
testing for common substring of length 4: found!
4


In [31]:
X = "0110011001101111"
Y = "1010011001100000"

print(quantum_LCS(X, Y, 8, 2, True))

testing for common substring of length 4: {'1': 21, '0': 4}
found!
testing for common substring of length 6: {'0': 25}
not found!
testing for common substring of length 5: {'1': 18, '0': 7}
found!
5


In [24]:
X = '0000000000000000'
Y = '1111111111111111'

print(quantum_LCS(X, Y, 8, 2, True))

testing for common substring of length 4: not found!
testing for common substring of length 2: not found!
testing for common substring of length 1: not found!
0


In [27]:
def classical_lcs_substring_len(x: str, y: str, char_size: int = 2) -> int:
    xs = [x[i:i+char_size] for i in range(0, len(x), char_size)]
    ys = [y[i:i+char_size] for i in range(0, len(y), char_size)]

    n = len(xs)
    m = len(ys)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    best = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if xs[i - 1] == ys[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                best = max(best, dp[i][j])
    return best


tests = [
    ("1010101000000000", "0101010111111111"),
    ("0101010000000000", "1010101111111111"),
    ("1010100000000000", "0101011111111111"),
    ("0101000000000000", "1010111111111111"),
    ("1001000000000000", "0110111111111111"),
    ("0110000000000000", "1001111111111111"),
    ("0010000000000000", "1101111111111111"),
    ("1000000000000000", "0111111111111111"),
    ("0100000000000000", "1011111111111111"),

    # atteso = 2
    ("0101010101000000", "1001011011111111"),
    ("1010101010000000", "0110100111111111"),
    ("0101101000000000", "1001011111111111"),
    ("1010010100000000", "0110101111111111"),
    ("0101001000000000", "1001011111111111"),
    ("1001010000000000", "0110101111111111"),
    ("0110010000000000", "1001101111111111"),
    ("0010100000000000", "1101011111111111"),
    ("1001100000000000", "0110011111111111"),
    ("0100100000000000", "1011011111111111"),

    # atteso = 3
    ("0101010110000000", "1001010111111111"),
    ("1010101001000000", "0110101011111111"),
    ("0101100100000000", "1001101111111111"),
    ("1010011000000000", "0110011111111111"),
    ("0101001010000000", "1001010111111111"),
    ("1001010110000000", "0110101011111111"),
    ("0110011010000000", "1001100111111111"),
    ("0010100110000000", "1101010111111111"),
    ("1001101010000000", "0110010111111111"),
    ("0100101010000000", "1011010111111111"),

    # atteso = 4
    ("0101010110000000", "1001010110111111"),
    ("1010101001000000", "0110101010111111"),
    ("0101100110000000", "1001100111111111"),
    ("1010011010000000", "0110011011111111"),
    ("0101001011000000", "1001001011111111"),
    ("1001010111000000", "0110100111111111"),
    ("0110011011000000", "1001100011111111"),
    ("0010100111000000", "1101010011111111"),
    ("1001101011000000", "0110010011111111"),
    ("0100101011000000", "1011010011111111"),

    # atteso = 5
    ("0101010110100000", "1001010110101111"),
    ("1010101001010000", "0110101001011111"),
    ("0101100110010000", "1001100110011111"),
    ("1010011001100000", "0110011001101111"),
    ("0101001011010000", "1001001011011111"),
    ("1001010110100000", "0110100110101111"),
    ("0110011010010000", "1001100010011111"),
    ("0010100110100000", "1101010010101111"),
    ("1001101010010000", "0110010010011111"),
    ("0100101011010000", "1011010011011111"),
    ("0010101001010000", "0010101001011111"),
    ("0101010010100000", "0101010010101111"),
    ("1001101001010000", "1001101001011111"),
    ("0110010101100000", "0110010101101111"),
    ("1010100110010000", "1010100110011111"),
    ("0010011001100000", "0010011001101111"),
    ("0101010101010100", "0101010101010111"),
    ("0101010101011000", "0101010101011011"),
    ("0101010110010100", "0101010110010111"),
    ("0101100101011000", "0101100101011011"),
    ("0101100110101000", "0101100110101011"),
    ("1001010101010100", "1001010101010111"),
    ("0110101001010000", "0110101001011111"),
    ("1001100110100000", "1001100110101111"),
    ("0101101001100000", "0101101001101111"),
    ("1010010110100000", "1010010110101111"),
    ("0010100101100000", "0010100101101111"),
    ("0101011001010000", "0101011001011111"),
    ("1001010010100000", "1001010010101111"),
    ("0110011001010000", "0110011001011111"),
    ("1010101001100000", "1010101001101111"),
    ("0010010110100000", "0010010110101111"),
    ("0101100110010000", "0101100110011111"),
    ("1001011001100000", "1001011001101111"),
    ("0110100110010000", "0110100110011111"),
    ("1010011001010000", "1010011001011111"),
    ("1001010110011000", "1001010110011011"),
    ("1001100101100100", "1001100101100111"),
    ("1001100110011000", "1001100110011011"),
    ("1010010101100000", "1010010101101111"),
    ("0110101001010000", "0110101001011111"),
    ("1001100110100000", "1001100110101111"),
    ("0101101001100000", "0101101001101111"),
    ("1010010110100000", "1010010110101111"),
    ("0010100101100000", "0010100101101111"),
    ("0101011001010000", "0101011001011111"),
    ("1001010010100000", "1001010010101111"),
    ("0110011001010000", "0110011001011111"),
    ("1010101001100000", "1010101001101111"),
    ("0010010110100000", "0010010110101111"),
    ("0101100110010000", "0101100110011111"),
    ("1001011001100000", "1001011001101111"),
    ("0110100110010000", "0110100110011111"),
    ("1010011001010000", "1010011001011111"),
    ("0010101001010000", "0010101001011111"),
    ("0101010010100000", "0101010010101111"),
    ("1001101001010000", "1001101001011111"),
    ("0110010101100000", "0110010101101111"),
    ("1010100110010000", "1010100110011111"),
    ("0010011001100000", "0010011001101111"),
]


passed = 0
failed = 0
for X, Y in tests:
    expected = classical_lcs_substring_len(X, Y, 2)
    print(f"X = '{X}'")
    print(f"Y = '{Y}'")
    actual = quantum_LCS(X, Y, 8, 2, False)
    print('atteso =', expected, '| ottenuto =', actual)

    if expected == actual:
        passed += 1
    else:
        failed += 1

print(passed)
print(failed)


X = '1010101000000000'
Y = '0101010111111111'
atteso = 0 | ottenuto = 0
X = '0101010000000000'
Y = '1010101111111111'
atteso = 0 | ottenuto = 0
X = '1010100000000000'
Y = '0101011111111111'
atteso = 0 | ottenuto = 0
X = '0101000000000000'
Y = '1010111111111111'


KeyboardInterrupt: 

In [26]:

tests_3bit = [
    ("001010011100101110000000", "001010011100101110111111"),
    ("010011100101110001010000", "010011100101110001010111"),
    ("011100101110001010000000", "011100101110001010111111"),
    ("100101110001010011100000", "100101110001010011100111"),
    ("101110001010011100000000", "101110001010011100111111"),
    ("110001010011100101000000", "110001010011100101111111"),
    ("001011101010100110000000", "001011101010100110111111"),
    ("010100110101001011000000", "010100110101001011111111"),
    ("011001101100010101000000", "011001101100010101111111"),
    ("100010101001011110000000", "100010101001011110111111"),
    ("101001011010100110000000", "101001011010100110111111"),
    ("110100101001101010000000", "110100101001101010111111"),
    ("001100101010110001000000", "001100101010110001111111"),
    ("010101001100110010000000", "010101001100110010111111"),
    ("011110100101001100000000", "011110100101001100111111"),
    ("100001011010110100000000", "100001011010110100111111"),
    ("101010100110001011000000", "101010100110001011111111"),
    ("110011001010100101000000", "110011001010100101111111"),
    ("001101010011001100000000", "001101010011001100111111"),
    ("010110001101010100000000", "010110001101010100111111"),
]

for X, Y in tests_3bit:
    print(f"X = '{X}'")
    print(f"Y = '{Y}'")
    expected = classical_lcs_substring_len(X, Y, 3)
    actual = quantum_LCS(X, Y, 8, 3, False)
    print("atteso =", expected, "| ottenuto =", actual)

    if expected == actual:
        passed += 1
    else:
        failed += 1

print(passed)
print(failed)


'\ntests_3bit = [\n    ("001010011100101110000000", "001010011100101110111111"),\n    ("010011100101110001010000", "010011100101110001010111"),\n    ("011100101110001010000000", "011100101110001010111111"),\n    ("100101110001010011100000", "100101110001010011100111"),\n    ("101110001010011100000000", "101110001010011100111111"),\n    ("110001010011100101000000", "110001010011100101111111"),\n    ("001011101010100110000000", "001011101010100110111111"),\n    ("010100110101001011000000", "010100110101001011111111"),\n    ("011001101100010101000000", "011001101100010101111111"),\n    ("100010101001011110000000", "100010101001011110111111"),\n    ("101001011010100110000000", "101001011010100110111111"),\n    ("110100101001101010000000", "110100101001101010111111"),\n    ("001100101010110001000000", "001100101010110001111111"),\n    ("010101001100110010000000", "010101001100110010111111"),\n    ("011110100101001100000000", "011110100101001100111111"),\n    ("100001011010110100000000", "10